In [17]:

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_CSV = ROOT / "data" / "raw" / "cicids2017_cleaned.csv"
SCHEMA_DIR = ROOT / "data" / "schema"

print("Dataset path:", RAW_CSV)
print("Dataset exists:", RAW_CSV.exists())


Dataset path: c:\Users\Admin\Documents\Kalasag\data\raw\cicids2017_cleaned.csv
Dataset exists: True


In [18]:

df = pd.read_csv(RAW_CSV, low_memory=False)

df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

display(df.head())


Dataset shape: (2520751, 53)
Total rows: 2520751
Total columns: 53


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [19]:

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nAttack Type column exists:", "Attack Type" in df.columns)


Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Max', 'Active Min', 'Idle Mean', '

In [20]:

class_counts = df["Attack Type"].value_counts()
class_percentages = df["Attack Type"].value_counts(normalize=True) * 100

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages.round(2)
})

print("Total attack classes:", df["Attack Type"].nunique())
print("\nClass distribution:")
display(class_distribution)


Total attack classes: 7

Class distribution:


,Count,Percentage
Attack Type,,
Normal Traffic,2095057,83.11
DoS,193745,7.69
DDoS,128014,5.08
Port Scanning,90694,3.60
Brute Force,9150,0.36
Web Attacks,2143,0.09
Bots,1948,0.08


In [21]:

numeric_df = df.select_dtypes(include=[np.number])

missing_values = df.isna().sum()
infinite_values = np.isinf(numeric_df).sum()

print("Total missing values:", missing_values.sum())
print("Total infinite values:", infinite_values.sum())

print("\nColumns with missing values:")
print(missing_values[missing_values > 0])

print("\nColumns with infinite values:")
print(infinite_values[infinite_values > 0])


Total missing values: 0
Total infinite values: 0

Columns with missing values:
Series([], dtype: int64)

Columns with infinite values:
Series([], dtype: int64)


In [22]:

duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("Total duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")


Total duplicate rows: 161
Duplicate percentage: 0.01 %


In [23]:

unique_counts = df.nunique(dropna=False)
constant_columns = unique_counts[unique_counts <= 1]

print("Total constant columns:", len(constant_columns))
print("\nConstant columns:")
print(constant_columns)


Total constant columns: 0

Constant columns:
Series([], dtype: int64)


In [24]:

numeric_df = df.select_dtypes(include=[np.number])

feature_ranges = pd.DataFrame({
    "Minimum": numeric_df.min(),
    "Maximum": numeric_df.max(),
    "Mean": numeric_df.mean(),
    "Standard Deviation": numeric_df.std()
})

display(feature_ranges.round(4))


,Minimum,Maximum,Mean,Standard Deviation
Destination Port,0.000000e+00,6.553500e+04,8.690744e+03,1.901294e+04
Flow Duration,-1.300000e+01,1.200000e+08,1.659032e+07,3.523140e+07
Total Fwd Packets,1.000000e+00,2.197590e+05,1.025880e+01,7.943824e+02
Total Length of Fwd Packets,0.000000e+00,1.290000e+07,6.065243e+02,1.011595e+04
Fwd Packet Length Max,0.000000e+00,2.482000e+04,2.311993e+02,7.563122e+02
Fwd Packet Length Min,0.000000e+00,2.325000e+03,1.920380e+01,6.079890e+01
Fwd Packet Length Mean,0.000000e+00,5.940857e+03,6.350190e+01,1.955512e+02
Fwd Packet Length Std,0.000000e+00,7.125597e+03,7.732030e+01,2.968824e+02
Bwd Packet Length Max,0.000000e+00,1.953000e+04,9.749253e+02,2.038152e+03
Bwd Packet Length Min,0.000000e+00,2.896000e+03,4.316020e+01,7.088020e+01


In [25]:

negative_counts = (numeric_df < 0).sum()
negative_columns = negative_counts[negative_counts > 0].sort_values(ascending=False)

negative_summary = pd.DataFrame({
    "Negative Count": negative_columns,
    "Percentage": (negative_columns / len(df) * 100).round(4),
    "Minimum Value": numeric_df[negative_columns.index].min()
})

display(negative_summary)

,Negative Count,Percentage,Minimum Value
Init_Win_bytes_backward,1215622,48.2246,-1.000000e+00
Init_Win_bytes_forward,911012,36.1405,-1.000000e+00
Flow IAT Min,2875,0.1141,-1.400000e+01
Flow Duration,107,0.0042,-1.300000e+01
Flow Packets/s,107,0.0042,-2.000000e+06
Flow IAT Mean,107,0.0042,-1.300000e+01
Flow IAT Max,107,0.0042,-1.300000e+01
Flow Bytes/s,78,0.0031,-2.610000e+08
min_seg_size_forward,35,0.0014,-5.368707e+08
Fwd Header Length,35,0.0014,-3.221223e+10


In [26]:

window_columns = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]

check_columns = negative_columns.index.difference(window_columns)

suspicious_rows = (df[check_columns] < 0).any(axis=1)

print("Rows with suspicious negative values:", suspicious_rows.sum())
print("Percentage:", round(suspicious_rows.mean() * 100, 4), "%")

print("\nAttack class distribution of suspicious rows:")
print(df.loc[suspicious_rows, "Attack Type"].value_counts())


Rows with suspicious negative values: 2910
Percentage: 0.1154 %

Attack class distribution of suspicious rows:
Attack Type
Normal Traffic    2722
DoS                163
DDoS                19
Brute Force          6
Name: count, dtype: int64


In [27]:

duplicate_rows = df.duplicated(keep="first")

print("Total duplicate rows:", duplicate_rows.sum())

print("\nDuplicate rows by attack class:")
print(df.loc[duplicate_rows, "Attack Type"].value_counts())


Total duplicate rows: 161

Duplicate rows by attack class:
Attack Type
Normal Traffic    161
Name: count, dtype: int64


In [28]:

SCHEMA_DIR.mkdir(parents=True, exist_ok=True)

report = "# CICIDS2017 Class Distribution\n\n"
report += f"Total records: {len(df):,}\n\n"
report += f"Total classes: {df['Attack Type'].nunique()}\n\n"
report += "| Attack Type | Count | Percentage |\n"
report += "|---|---:|---:|\n"

for attack_type, row in class_distribution.iterrows():
    report += f"| {attack_type} | {int(row['Count']):,} | {row['Percentage']:.2f}% |\n"

report += "\nThe dataset is highly imbalanced, with Normal Traffic representing 83.11% of records.\n"

report_path = SCHEMA_DIR / "class_distribution.md"
report_path.write_text(report, encoding="utf-8")

print("Report saved:", report_path)
print("File exists:", report_path.exists())


Report saved: c:\Users\Admin\Documents\Kalasag\data\schema\class_distribution.md
File exists: True


In [30]:

feature_columns = [col for col in df.columns if col != "Attack Type"]

schema = {
    "version": 1,
    "dataset": "CICIDS2017 Cleaned and Preprocessed",
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "label_column": "Attack Type",
    "label_classes": sorted(df["Attack Type"].unique().tolist()),
    "excluded_columns": [],
    "feature_count": len(feature_columns),
    "features": [
        {
            "name": col,
            "dtype": str(df[col].dtype),
            "description": ""
        }
        for col in feature_columns
    ],
    "scaler_file": "models/scaler.pkl",
    "label_encoder_file": "models/label_encoder.pkl"
}

schema_path = SCHEMA_DIR / "feature_schema.json"
schema_path.parent.mkdir(parents=True, exist_ok=True)

with schema_path.open("w", encoding="utf-8") as file:
    json.dump(schema, file, indent=4)

print("Schema updated:", schema_path)
print("Feature count:", schema["feature_count"])
print("Label column:", schema["label_column"])
print("Label classes:", schema["label_classes"])


Schema updated: c:\Users\Admin\Documents\Kalasag\data\schema\feature_schema.json
Feature count: 52
Label column: Attack Type
Label classes: ['Bots', 'Brute Force', 'DDoS', 'DoS', 'Normal Traffic', 'Port Scanning', 'Web Attacks']


In [ ]:

with schema_path.open("r", encoding="utf-8") as file:
    saved_schema = json.load(file)

print("Version:", saved_schema["version"])
print("Label column:", saved_schema["label_column"])
print("Feature count:", saved_schema["feature_count"])
print("Label classes:", saved_schema["label_classes"])
print("Excluded columns:", saved_schema["excluded_columns"])
print("Scaler file:", saved_schema["scaler_file"])
print("Label encoder file:", saved_schema["label_encoder_file"])
print("First five features:", saved_schema["features"][:5])


NameError: name 'schema_path' is not defined

In [ ]:

audit = f"""# CICIDS2017 Data Quality Audit

## Dataset Overview
- Total records: {len(df):,}
- Total columns: {len(df.columns)}
- Input features: {len(feature_columns)}
- Target column: Attack Type
- Traffic classes: {df["Attack Type"].nunique()}

## Quality Checks
- Missing values: {int(missing_values.sum()):,}
- Infinite values: {int(infinite_values.sum()):,}
- Duplicate records: {int(duplicate_count):,}
- Constant columns: {len(constant_columns)}
- Records with suspicious negative values: {int(suspicious_rows.sum()):,}

## Findings
- Normal Traffic represents 83.11% of the dataset.
- All 161 duplicate records belong to Normal Traffic.
- Some timing, rate, header-length, and segment-size features contain negative values.
- TCP window-size features contain -1 values that may represent undefined measurements.
- Numeric features have substantially different ranges and do not appear standardized.

## Preprocessing Considerations
- Review suspicious negative values before deciding whether to remove or correct records.
- Evaluate duplicate removal before model training.
- Split the dataset before fitting a scaler.
- Fit scaling parameters using training data only.
- Apply SMOTE only to the training partition, if selected.
- Preserve the original CSV.
"""

audit_path = SCHEMA_DIR / "data_quality_audit.md"
audit_path.write_text(audit, encoding="utf-8")

print("Audit saved:", audit_path)
print("File exists:", audit_path.exists())


In [31]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_CSV = ROOT / "data" / "raw" / "cicids2017_cleaned.csv"
SCHEMA_DIR = ROOT / "data" / "schema"

LABEL = "Attack Type"

df = pd.read_csv(RAW_CSV)
df.columns = df.columns.str.strip()

print("Dataset loaded:", df.shape)
print("Label exists:", LABEL in df.columns)

Dataset loaded: (2520751, 53)
Label exists: True


In [32]:
rows_before = len(df)
columns_before = len(df.columns)
before_counts = df[LABEL].value_counts()

print("Rows before cleaning:", rows_before)
print("Columns before cleaning:", columns_before)
print(before_counts)

Rows before cleaning: 2520751
Columns before cleaning: 53
Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64


In [33]:
num_cols = df.select_dtypes(include=[np.number]).columns

infinite_count = np.isinf(df[num_cols].to_numpy()).sum()

df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

bad_rows = df.isna().any(axis=1)

print("Infinite values replaced:", infinite_count)
print("Rows with missing values:", bad_rows.sum())
print(df.loc[bad_rows, LABEL].value_counts())

Infinite values replaced: 0
Rows with missing values: 0
Series([], Name: count, dtype: int64)


In [34]:
feature_cols = [c for c in df.columns if c != LABEL]

print("Exact duplicates:", df.duplicated().sum())
print(df.loc[df.duplicated(), LABEL].value_counts())
print("Feature-only duplicates:", df.duplicated(subset=feature_cols).sum())

df = df.drop_duplicates().copy()
gc.collect()

Exact duplicates: 161
Attack Type
Normal Traffic    161
Name: count, dtype: int64
Feature-only duplicates: 858


883

In [35]:
feature_cols = [c for c in df.columns if c != LABEL]

const_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]

print("Constant columns:", const_cols)

df = df.drop(columns=const_cols)

num_now = df.select_dtypes(include=[np.number]).columns

print("Non-numeric features:", [c for c in df.columns if c != LABEL and c not in num_now])
print("ID-like columns:", [c for c in df.columns if any(k in c.lower() for k in ["flow id", "ip", "timestamp", "date"])])

Constant columns: []
Non-numeric features: []
ID-like columns: []


In [36]:
df[LABEL] = df[LABEL].astype(str).str.strip()

NORMAL = "Normal Traffic"

print(sorted(df[LABEL].unique()))
assert NORMAL in set(df[LABEL])

after_counts = df[LABEL].value_counts()

compare = pd.DataFrame({
    "before": before_counts,
    "after": after_counts
})

compare["removed"] = compare["before"] - compare["after"]
compare["removed_pct"] = (
    compare["removed"] / compare["before"] * 100
).round(3)

print("Rows:", rows_before, "->", len(df))
print("Columns:", columns_before, "->", len(df.columns))

compare

['Bots', 'Brute Force', 'DDoS', 'DoS', 'Normal Traffic', 'Port Scanning', 'Web Attacks']
Rows: 2520751 -> 2520590
Columns: 53 -> 53


,before,after,removed,removed_pct
Attack Type,,,,
Normal Traffic,2095057,2094896,161,0.008
DoS,193745,193745,0,0.000
DDoS,128014,128014,0,0.000
Port Scanning,90694,90694,0,0.000
Brute Force,9150,9150,0,0.000
Web Attacks,2143,2143,0,0.000
Bots,1948,1948,0,0.000


In [37]:
schema_path = SCHEMA_DIR / "feature_schema.json"
schema = json.loads(schema_path.read_text(encoding="utf-8"))

schema["excluded_columns"] = const_cols
schema["features"] = [
    {"name": c, "dtype": str(df[c].dtype)}
    for c in df.columns
    if c != LABEL
]
schema["version"] += 1

schema_path.write_text(
    json.dumps(schema, indent=2),
    encoding="utf-8"
)

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

df.to_parquet(INTERIM / "cleaned.parquet", index=False)

print("Saved cleaned data:", df.shape)

del df
gc.collect()

Saved cleaned data: (2520590, 53)


0